<a href="https://colab.research.google.com/github/Sanjaysanjay31/24B11DS095_DriveReady_OPC_2026/blob/main/Day_2/Startup_Marketing_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2 — Prompts, Memory, and Python Fundamentals for Agents

**Project:** AI Startup Marketing Agent  
**Curriculum:** Aditya University — One Person Company (12-Day Agentic AI Workshop)  
**Primary Provider:** OpenRouter — NVIDIA Nemotron 3.5 Super VL (Free)  
**Fallback Provider:** Groq — GPT-OSS 120B  
**Case Study:** Autonomous 9:16 Video Marketing Assistant for Early-Stage Startups  

This notebook implements Day 2 fundamentals: messages and roles, structured system prompts, few-shot classification, JSON data extraction, sliding window conversation memory, and essential Python primitives (dictionaries, type hints, decorators, try/except). It supports execution both in Google Colab (via Colab Secrets) and locally (via `.env` file).

In [1]:
# Install the official OpenAI client and python-dotenv for local and Colab execution
!pip install -q openai python-dotenv

In [2]:
import os
import json
from openai import OpenAI

# 1. Load credentials from local .env file (for local Jupyter / VS Code runs)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 2. Retrieve credentials from Google Colab Secrets if running in Colab
try:
    from google.colab import userdata
    colab_openrouter = userdata.get("OPENROUTER_API_KEY")
    colab_groq = userdata.get("GROQ_API_KEY") or userdata.get("agentic_ai_GROQ_API_KEY")
except Exception:
    colab_openrouter = None
    colab_groq = None

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or colab_openrouter
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or colab_groq

# Primary Provider: OpenRouter (NVIDIA Nemotron 3.5 Super VL)
openrouter_client = None
if OPENROUTER_API_KEY:
    openrouter_client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1"
    )

# Fallback Provider: Groq (GPT-OSS 120B)
groq_client = None
if GROQ_API_KEY:
    groq_client = OpenAI(
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1"
    )

PRIMARY_MODEL = "nvidia/nemotron-3.5-super-vl:free"
FALLBACK_MODEL = "openai/gpt-oss-120b"

print("AI Startup Marketing Assistant API setup completed!")

AI Startup Marketing Assistant API setup completed!


In [3]:
# Marketing service catalogue with pricing in Indian Rupees (INR)
MARKETING_SERVICES = {
    "viral_hook_generation": 400,
    "short_script_writing": 700,
    "storyboard_generation": 1200,
    "voiceover_synthesis": 500,
    "auto_subtitle_karaoke": 300,
    "vertical_video_assembly": 1500,
    "seo_metadata_pack": 350
}

print("Available Startup Marketing Services:\n")
for service, price in MARKETING_SERVICES.items():
    print(f"{service.replace('_', ' ').title()} : Rs {price}")

Available Startup Marketing Services:

Viral Hook Generation : Rs 400
Short Script Writing : Rs 700
Storyboard Generation : Rs 1200
Voiceover Synthesis : Rs 500
Auto Subtitle Karaoke : Rs 300
Vertical Video Assembly : Rs 1500
Seo Metadata Pack : Rs 350


In [4]:
# System prompt establishing Role, Facts, Rules, and Output format
MARKETING_SYSTEM_PROMPT = """
You are an intelligent AI Startup Marketing Assistant.

Your role is to help early-stage startup founders create high-converting,
faceless short-form videos for YouTube Shorts and Instagram Reels.

Services and pricing:
- Viral hook generation: Rs 400
- Short script writing: Rs 700
- Storyboard generation: Rs 1200
- Voiceover synthesis: Rs 500
- Auto subtitle karaoke: Rs 300
- Vertical video assembly: Rs 1500
- SEO metadata pack: Rs 350

Rules:
- Greet founders politely and understand their core product value.
- Focus video angles on real customer pain points rather than generic product features.
- Provide service prices strictly from the approved catalogue.
- If a requested service is not in our list, clarify politely.
- Never guarantee viral views, exact follower counts, or overnight revenue.
- Never invent fictional statistics, testimonials, or partnerships.
- Clarify that video rendering is handled asynchronously after script approval.
- Format responses concisely (under 60 words) and keep tone practical and encouraging.
"""

print("System prompt created!")

System prompt created!


In [5]:
# LLM invocation function with automatic primary-to-fallback routing
def ask_llm(messages):
    """
    Attempts call with primary OpenRouter model (NVIDIA Nemotron 3.5 Super VL).
    Automatically fails over to Groq (GPT-OSS 120B) if quota or connection fails.
    """
    # 1. Attempt primary provider (OpenRouter)
    if openrouter_client is not None:
        try:
            response = openrouter_client.chat.completions.create(
                model=PRIMARY_MODEL,
                messages=messages,
                temperature=0.3
            )
            return response.choices[0].message.content
        except Exception as error:
            print(f"[Failover] OpenRouter failed ({type(error).__name__}). Switching to Groq fallback.")

    # 2. Attempt fallback provider (Groq)
    if groq_client is not None:
        response = groq_client.chat.completions.create(
            model=FALLBACK_MODEL,
            messages=messages,
            temperature=0.3
        )
        return response.choices[0].message.content

    raise RuntimeError("No active LLM provider. Configure OPENROUTER_API_KEY or GROQ_API_KEY in .env or Colab Secrets.")

print("LLM function created with OpenRouter primary and Groq fallback!")

LLM function created with OpenRouter primary and Groq fallback!


In [6]:
# Single-turn chat helper combining system prompt and user input
def marketing_chat(user_message):
    messages = [
        {"role": "system", "content": MARKETING_SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    return ask_llm(messages)

print(marketing_chat("Hello, I built a B2B SaaS tool for automated invoicing and need marketing help."))

Welcome! For an automated invoicing SaaS, we can craft high-retention 9:16 video hooks focusing on late payments and lost accounting hours. We offer viral hook generation (Rs 400) and short-form scriptwriting (Rs 700) to turn your workflow into relatable founder content. What is your primary target audience?


In [7]:
# Verify system prompt compliance when queried about services and pricing
print(marketing_chat("What services do you provide for startup video marketing?"))

We provide end-to-end short-form video marketing services: Viral Hook Generation (Rs 400), Short Script Writing (Rs 700), Storyboard Generation (Rs 1200), Voiceover Synthesis (Rs 500), Auto Subtitle Karaoke (Rs 300), Vertical Video Assembly (Rs 1500), and SEO Metadata Packs (Rs 350). All tailored for faceless 9:16 Shorts and Reels!


In [8]:
# Few-shot prompt to categorize incoming founder inquiries into distinct intents
CLASSIFICATION_PROMPT = """
You are a startup marketing request classifier.

Classify the customer's message into exactly one category:

hook_creation
scriptwriting
storyboard
voiceover
video_assembly
pricing_inquiry
other

Examples:

Customer: I need an opening line that stops developers from scrolling past my tool.
Category: hook_creation

Customer: Can you write a 30-second script showing why manual expense reporting wastes time?
Category: scriptwriting

Customer: I need visual scene prompts and image descriptions for a 4-scene video.
Category: storyboard

Customer: Can you generate a natural American English voiceover for my script?
Category: voiceover

Customer: Please assemble my audio, subtitles, and images into a 9:16 Reel.
Category: video_assembly

Customer: How much does it cost to get a full video assembled?
Category: pricing_inquiry

Customer: What is the weather today?
Category: other

Return only the category name.
"""

def classify_marketing_request(message):
    messages = [
        {"role": "system", "content": CLASSIFICATION_PROMPT},
        {"role": "user", "content": message}
    ]
    return ask_llm(messages).strip().lower()

print(classify_marketing_request("I need an aggressive hook to grab early adopters for my dev tool."))
print(classify_marketing_request("How much does the voiceover synthesis cost?"))
print(classify_marketing_request("Combine my voice track and frames into a vertical video."))

hook_creation
pricing_inquiry
video_assembly


In [9]:
# Extraction helper to parse unstructured founder messages into typed JSON data
def extract_startup_details(message):
    prompt = """
    Extract the following information from the customer message:

    {
        "founder_name": "",
        "startup_name": "",
        "product_domain": "",
        "services_requested": [],
        "target_problem": ""
    }

    Rules:
    - Return valid JSON only.
    - Use empty strings for missing text fields.
    - Use an empty list if services are not mentioned.
    - Do not invent missing information.
    """

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": message}
    ]

    result = ask_llm(messages)

    # Clean markdown fences if returned by the LLM
    result = result.strip()
    if result.startswith("```"):
        result = result.replace("```json", "", 1)
        result = result.replace("```", "")
        result = result.strip()

    return json.loads(result)

founder_brief = """
My name is Sanjay.
I run DevPulse, an AI code review assistant for engineering teams.
I need viral hook generation and short script writing.
Developers are ignoring pull requests and shipping bugs due to fatigue.
"""

extracted_brief = extract_startup_details(founder_brief)
print(json.dumps(extracted_brief, indent=4))

{
    "founder_name": "Sanjay",
    "startup_name": "DevPulse",
    "product_domain": "AI code review assistant",
    "services_requested": [
        "viral_hook_generation",
        "short_script_writing"
    ],
    "target_problem": "Developers are ignoring pull requests and shipping bugs due to fatigue"
}


In [10]:
# Cost calculation function with strict validation and error handling
def calculate_service_cost(services):
    total = 0
    for service in services:
        if service not in MARKETING_SERVICES:
            raise ValueError(f"Unknown service: {service}")
        total += MARKETING_SERVICES[service]
    return total

selected_services = [
    "viral_hook_generation",
    "short_script_writing",
    "storyboard_generation"
]

total_cost = calculate_service_cost(selected_services)

print("Selected Campaign Services:")
for service in selected_services:
    print(service.replace("_", " ").title())

print("Total Estimated Cost: Rs", total_cost)

Selected Campaign Services:
Viral Hook Generation
Short Script Writing
Storyboard Generation
Total Estimated Cost: Rs 2300


In [11]:
# Decorator to inspect and audit function execution
def log_execution(func):
    def wrapper(*args, **kwargs):
        print(f"Executing: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Completed: {func.__name__}")
        return result
    return wrapper

@log_execution
def calculate_marketing_bill(services):
    return calculate_service_cost(services)

bill = calculate_marketing_bill([
    "viral_hook_generation",
    "voiceover_synthesis",
    "vertical_video_assembly"
])

print("Estimated Campaign Bill: Rs", bill)

Executing: calculate_marketing_bill
Completed: calculate_marketing_bill
Estimated Campaign Bill: Rs 2400


In [12]:
# Robust error handling wrapper guarding against API rate limits and network outages
from openai import RateLimitError, AuthenticationError, APIConnectionError

def safe_marketing_chat(message):
    try:
        return marketing_chat(message)
    except RateLimitError:
        return "Server is busy. Please try again later."
    except AuthenticationError:
        return "API key is invalid. Please check your setup in .env or Colab Secrets."
    except APIConnectionError:
        return "Connection problem. Check your internet connection."
    except Exception as error:
        return f"An unexpected error occurred: {error}"

In [13]:
# Chat buffer implementing sliding window memory to retain conversation context
marketing_history = []
MAX_TURNS = 5

def marketing_chat_with_memory(user_message):
    marketing_history.append({
        "role": "user",
        "content": user_message
    })

    # Maintain the latest 5 user-assistant exchanges to prevent context bloat
    recent_history = marketing_history[-(MAX_TURNS * 2):]

    messages = [
        {"role": "system", "content": MARKETING_SYSTEM_PROMPT}
    ] + recent_history

    try:
        answer = ask_llm(messages)
        marketing_history.append({
            "role": "assistant",
            "content": answer
        })
        return answer
    except Exception:
        marketing_history.pop()
        raise

print(marketing_chat_with_memory("My name is Sanjay."))
print(marketing_chat_with_memory("I run DevPulse, an AI code review assistant."))
print(marketing_chat_with_memory("What is my name and what is my startup?"))

Hello Sanjay! Nice to meet you. How can I help you market your startup today?
DevPulse sounds like a valuable tool for software engineering teams. We can create compelling hooks and scripts addressing developer burnout and bug leaks!
Your name is Sanjay, and you run DevPulse, an AI code review assistant!


In [14]:
# Automated multi-turn dialogue verification
test_messages = [
    "Hello, my name is Sanjay.",
    "I run DevPulse, an AI code review assistant.",
    "Developers are ignoring pull requests and shipping bugs.",
    "What services can help me turn this into a short video?",
    "How much does short script writing cost?",
    "What startup do I run?"
]

for message in test_messages:
    print("\nYou:", message)
    response = marketing_chat_with_memory(message)
    print("Marketing Assistant:", response)


You: Hello, my name is Sanjay.
Marketing Assistant: Hello Sanjay! Welcome. How can I help you market your startup today?

You: I run DevPulse, an AI code review assistant.
Marketing Assistant: Great to know about DevPulse! We can create high-impact 9:16 video hooks focusing on catching bugs before production.

You: Developers are ignoring pull requests and shipping bugs.
Marketing Assistant: That is a sharp, relatable pain point. A hook like 'Why your team is approving PRs without reading them' will stop developers scrolling.

You: What services can help me turn this into a short video?
Marketing Assistant: I recommend Viral Hook Generation (Rs 400), Short Script Writing (Rs 700), and Vertical Video Assembly (Rs 1500) for a complete Reel.

You: How much does short script writing cost?
Marketing Assistant: Short Script Writing costs Rs 700.

You: What startup do I run?
Marketing Assistant: You run DevPulse, an AI code review assistant for developers!


In [15]:
# Interactive terminal chat loop with graceful termination
print("Welcome to AI Startup Marketing Assistant!")
print("Ask me about viral hooks, short scripts, storyboards, and pricing.")
print("Type 'bye' to exit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower().strip() == "bye":
        print("Marketing Assistant: Thank you for visiting. Best of luck with your startup launch!")
        break

    if not user_input.strip():
        print("Marketing Assistant: Please enter a message.")
        continue

    response = safe_marketing_chat(user_input)
    print("Marketing Assistant:", response)